# EcoScan: Micro-Plastic Detection using Attention-Augmented CNNs
**Assignment 2 — Data and Method Implementation**  
**Monish | a1994640 | Deep Learning Applications**

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU** → Save
2. Then run all cells top to bottom (**Runtime → Run all**)

Full pipeline (~2–3 hours on T4 GPU):
- Step 1: Install dependencies
- Step 2: Clone EcoScan repo from GitHub
- Step 3: Generate 10,000-image Floating Debris dataset
- Step 4: Train Baseline YOLOv8n (150 epochs)
- Step 5: Train CBAM-YOLOv8 (150 epochs)
- Step 6: Evaluate & generate comparison plots

## Step 1 — Check GPU & Install Dependencies

In [ ]:
# Verify GPU is available
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install all dependencies
!pip install ultralytics opencv-python albumentations seaborn matplotlib roboflow -q
print("Dependencies installed.")

## Step 2 — Clone EcoScan from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/saravmonish/ecoscan.git"
PROJECT_DIR = "/content/ecoscan"

if os.path.exists(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
!ls

## Step 3 — Generate Floating Debris Dataset (10,000 images)
Exact class distribution from the assignment PDF:
- plastic_bottle: **3,420** | plastic_bag: **2,180** | foam_styrofoam: **1,650**
- fishing_net: **1,280** | other_debris: **980** | micro_plastic: **490**

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from utils.dataset_generator import generate_dataset
from utils.preprocessing import preprocess_dataset
from utils.evaluate import plot_class_distribution, plot_augmentation_samples, CLASS_NAMES

DATA_DIR    = os.path.join(PROJECT_DIR, "data", "floating_debris")
OUTPUT_DIR  = os.path.join(PROJECT_DIR, "outputs")
RUNS_DIR    = os.path.join(PROJECT_DIR, "runs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RUNS_DIR,   exist_ok=True)

import time
t = time.time()
yaml_path = generate_dataset(DATA_DIR, n_train=7000, n_val=1500, n_test=1500, img_size=640)
print(f"\nDataset generated in {(time.time()-t)/60:.1f} min")
print(f"YAML: {yaml_path}")

In [ ]:
# Apply CLAHE preprocessing
print("Applying CLAHE preprocessing...")
preprocess_dataset(DATA_DIR)
print("Done.")

In [ ]:
# Visualise class distribution
from IPython.display import Image as IPImage, display
plot_class_distribution(DATA_DIR, OUTPUT_DIR)
plot_augmentation_samples(DATA_DIR, OUTPUT_DIR)
display(IPImage(os.path.join(OUTPUT_DIR, 'class_distribution.png')))

## Step 4 — Train Baseline YOLOv8n

In [ ]:
from models.cbam_yolov8 import train_baseline_yolov8

device = "0" if torch.cuda.is_available() else "cpu"
print(f"Training on: {device}")

t = time.time()
baseline_model, baseline_results = train_baseline_yolov8(
    dataset_yaml=yaml_path,
    project_dir=RUNS_DIR,
    epochs=150,
    batch_size=16,
    imgsz=640,
    device=device,
)
print(f"\nBaseline training complete in {(time.time()-t)/60:.1f} min")

## Step 5 — Train CBAM-YOLOv8 (Proposed Architecture)

In [ ]:
from models.cbam_yolov8 import train_cbam_yolov8

t = time.time()
cbam_model, cbam_results = train_cbam_yolov8(
    dataset_yaml=yaml_path,
    project_dir=RUNS_DIR,
    epochs=150,
    batch_size=16,
    imgsz=640,
    device=device,
)
print(f"\nCBAM training complete in {(time.time()-t)/60:.1f} min")

## Step 6 — Evaluate & Compare Both Models

In [ ]:
from utils.evaluate import (
    plot_map_comparison, plot_per_class_metrics,
    plot_training_comparison, generate_results_summary
)

# Evaluate baseline
print("Evaluating Baseline YOLOv8...")
baseline_val = baseline_model.val(data=yaml_path, split="test", device=device, verbose=False)

# Evaluate CBAM
print("Evaluating CBAM-YOLOv8...")
cbam_val = cbam_model.val(data=yaml_path, split="test", device=device, verbose=False)

def extract_metrics(val_results):
    metrics = {}
    metrics["mAP@0.5"]       = float(val_results.box.map50)
    metrics["mAP@0.5:0.95"]  = float(val_results.box.map)
    metrics["Precision"]     = float(val_results.box.mp)
    metrics["Recall"]        = float(val_results.box.mr)
    p, r = metrics["Precision"], metrics["Recall"]
    metrics["F1"] = 2 * p * r / (p + r) if (p + r) > 0 else 0
    if hasattr(val_results.box, 'ap50') and val_results.box.ap50 is not None:
        for i, name in enumerate(CLASS_NAMES):
            if i < len(val_results.box.ap50):
                metrics[f"AP_{name}"] = float(val_results.box.ap50[i])
    if hasattr(val_results.box, 'p') and val_results.box.p is not None:
        p_list, r_list, f1_list = [], [], []
        for i in range(min(6, len(val_results.box.p))):
            pi, ri = float(val_results.box.p[i]), float(val_results.box.r[i])
            p_list.append(pi); r_list.append(ri)
            f1_list.append(2*pi*ri/(pi+ri) if (pi+ri) > 0 else 0)
        metrics["per_class_P"]  = p_list
        metrics["per_class_R"]  = r_list
        metrics["per_class_F1"] = f1_list
    return metrics

baseline_metrics = extract_metrics(baseline_val)
cbam_metrics     = extract_metrics(cbam_val)
generate_results_summary(baseline_metrics, cbam_metrics, OUTPUT_DIR)

In [ ]:
# Generate comparison plots
import os

plot_map_comparison(baseline_metrics, cbam_metrics, OUTPUT_DIR)

if "per_class_P" in baseline_metrics and "per_class_P" in cbam_metrics:
    results_dict = {
        "Baseline YOLOv8":    {"Precision": baseline_metrics["per_class_P"],
                               "Recall":    baseline_metrics["per_class_R"],
                               "F1":        baseline_metrics["per_class_F1"]},
        "CBAM-YOLOv8 (Ours)": {"Precision": cbam_metrics["per_class_P"],
                               "Recall":    cbam_metrics["per_class_R"],
                               "F1":        cbam_metrics["per_class_F1"]},
    }
    plot_per_class_metrics(results_dict, OUTPUT_DIR)

baseline_csv = os.path.join(RUNS_DIR, "baseline_yolov8", "results.csv")
cbam_csv     = os.path.join(RUNS_DIR, "cbam_yolov8",     "results.csv")
if os.path.exists(baseline_csv) and os.path.exists(cbam_csv):
    plot_training_comparison(baseline_csv, cbam_csv, OUTPUT_DIR)

print("All plots saved to:", OUTPUT_DIR)
for f in os.listdir(OUTPUT_DIR):
    print(" ", f)

In [ ]:
# Display all output plots
from IPython.display import Image as IPImage, display
for plot in ['class_distribution.png', 'map_comparison.png',
             'per_class_metrics.png', 'training_comparison.png']:
    path = os.path.join(OUTPUT_DIR, plot)
    if os.path.exists(path):
        print(f"\n--- {plot} ---")
        display(IPImage(path))

## Step 7 — Download Results

In [ ]:
# Zip and download all outputs + trained weights
import shutil

shutil.make_archive('/content/ecoscan_results', 'zip', '/content/ecoscan/outputs')

# Also zip the trained weights
os.makedirs('/content/weights_tmp', exist_ok=True)
import glob
for wf in glob.glob('/content/ecoscan/runs/**/best.pt', recursive=True):
    model_name = wf.split('/')[-3]
    shutil.copy(wf, f'/content/weights_tmp/{model_name}_best.pt')
shutil.make_archive('/content/ecoscan_weights', 'zip', '/content/weights_tmp')

from google.colab import files
files.download('/content/ecoscan_results.zip')
files.download('/content/ecoscan_weights.zip')
print("\nDownloads started!")
print("  ecoscan_results.zip  → all plots + results_summary.txt")
print("  ecoscan_weights.zip  → baseline_best.pt + cbam_best.pt")